# Silver Layer — CRM: Sales Details

Transforms the bronze `workspace.bronze.sales_details` table into `workspace.silver.sales_details` with meaningful, readable column names and cleaned data.

---

## Table of Contents — Transformations Applied

| # | Transformation | Description |
|---|---|---|
| 1 | **Column Renaming** | Renamed abbreviated columns to meaningful names (e.g., `sls_ord_num` → `order_number`, `sls_cust_id` → `customer_id`) |
| 2 | **Date Conversion** | Converted `order_date`, `ship_date`, `due_date` from integer (YYYYMMDD format) to actual DATE type. `order_date` with value `0` (17 rows) set to NULL |
| 3 | **NULL Sales Handling** | Replaced NULL `sales_amount` values with `0` |
| 4 | **NULL Price Handling** | Replaced NULL `price` values with `0` |
| 5 | **Negative Sales Handling** | Replaced negative `sales_amount` values (min was -54) with `0` |
| 6 | **Negative Price Handling** | Replaced negative `price` values (min was -1701) with `0` |

---

## Column Mapping (Bronze → Silver)

| Bronze Column | Silver Column | Transformation Applied |
|---|---|---|
| `sls_ord_num` | `order_number` | Renamed |
| `sls_prd_key` | `product_key` | Renamed |
| `sls_cust_id` | `customer_id` | Renamed |
| `sls_order_dt` | `order_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_ship_dt` | `ship_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_due_dt` | `due_date` | Renamed + Converted from int (YYYYMMDD) to DATE |
| `sls_sales` | `sales_amount` | Renamed + NULLs → 0 + Negatives → 0 |
| `sls_quantity` | `quantity` | Renamed |
| `sls_price` | `price` | Renamed + NULLs → 0 + Negatives → 0 |

---

## Implementation

All transformations are implemented in **PySpark** (`pyspark.sql.functions`):
- `col()` — select and alias columns
- `to_date()` — convert integer dates (YYYYMMDD) to actual DATE type
- `coalesce()` — replace NULL values with 0
- `when()` + `otherwise()` — replace negative values with 0
- `df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable()` — write to silver layer

In [0]:
from pyspark.sql.functions import col, to_date, coalesce, lit, when, length

df = (spark.read.table("workspace.bronze.sales_details")
    .select(
        col("sls_ord_num").alias("order_number"),
        col("sls_prd_key").alias("product_key"),
        col("sls_cust_id").alias("customer_id"),
        when(length(col("sls_order_dt").cast("string")) == 8, to_date(col("sls_order_dt").cast("string"), "yyyyMMdd")).otherwise(lit(None).cast("date")).alias("order_date"),
        to_date(col("sls_ship_dt").cast("string"), "yyyyMMdd").alias("ship_date"),
        to_date(col("sls_due_dt").cast("string"), "yyyyMMdd").alias("due_date"),
        when(coalesce(col("sls_sales"), lit(0)) < 0, 0)
            .otherwise(coalesce(col("sls_sales"), lit(0))).alias("sales_amount"),
        col("sls_quantity").alias("quantity"),
        when(coalesce(col("sls_price"), lit(0)) < 0, 0)
            .otherwise(coalesce(col("sls_price"), lit(0))).alias("price"),
    )
)

df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.silver.sales_details")

In [0]:
display(spark.read.table("workspace.silver.sales_details").limit(10))